# Experiment: Raw Candle History vs Pre-Computed Indicators

**Question:** Can a DNN learn better patterns from raw OHLCV of the last 21 candles than from pre-computed indicators (RSI, ADX, Bollinger, etc.)?

**Comparison (same ContextConditionedTCN architecture):**
- **A) Current v2:** 11 raw snapshot + 22 pre-computed indicators = 33 features
- **B) Raw candle history:** 11 raw snapshot + 21×5 prior candle OHLCV = 116 features
- **C) Combined:** 11 raw snapshot + 22 indicators + 21×5 OHLCV = 138 features

**Evaluation:** Actual trading strategy (edge-based `run_scaling` with fees), NOT raw accuracy.

All models trained on 80%, evaluated on unseen 20%.

In [1]:
import json
import sqlite3
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "..")
sys.path.insert(0, "../..")
from polybot.adapters.dnn_models import ContextConditionedTCN
from strategy_engine import StrategyConfig, run_scaling

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]

CROSS_CANDLE_COLS = [
    "prior_return",
    "consecutive_streak",
    "streak_magnitude",
    "rolling_volatility",
    "candle_momentum",
    "ma_crossover",
    "trend_consistency",
    "reversal_regime",
    "rsi",
    "bollinger_pct_b",
    "stochastic_k",
    "adx",
    "return_autocorrelation",
    "multi_candle_return_3",
    "multi_candle_return_6",
    "regime_score",
    "mean_reversion_signal",
    "prior_reversal_rate",
    "volume_momentum",
    "volume_trend",
    "volume_price_correlation",
    "relative_volume",
]

# Prior candle OHLCV features: 21 candles x 5 features = 105 columns
CANDLE_LOOKBACK = 21
CANDLE_FEATURES = ["open", "high", "low", "close", "volume"]

ELAPSED_CUTOFF = 0.50

## 1. Build Datasets

Load snapshots from JSONL, load candle OHLCV from DB, build 3 feature sets.

In [2]:
# Load snapshots
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# Load candle OHLCV from DB
conn = sqlite3.connect("file:../../data/collection.db?mode=ro", uri=True)
candles_db = pd.read_sql(
    "SELECT candle_id, open, high, low, close, volume, outcome FROM candles ORDER BY start_time", conn
)
conn.close()

# Build candle_id → ordered list of prior candle OHLCV
candle_order = candles_db["candle_id"].tolist()
candle_ohlcv = candles_db[CANDLE_FEATURES].values.astype(np.float32)

# For each candle, get the 21 prior candles' OHLCV as a flat vector
candle_id_to_idx = {cid: i for i, cid in enumerate(candle_order)}


def get_prior_ohlcv(candle_id):
    """Return flat array of [21 x 5] = 105 features for the 21 candles before this one."""
    idx = candle_id_to_idx.get(candle_id)
    if idx is None or idx < CANDLE_LOOKBACK:
        return None
    prior = candle_ohlcv[idx - CANDLE_LOOKBACK : idx]  # (21, 5)
    return prior.flatten()  # (105,)


# Build column names for prior candle features
PRIOR_CANDLE_COLS = [f"candle_{i}_{feat}" for i in range(CANDLE_LOOKBACK) for feat in CANDLE_FEATURES]

print(f"Snapshots: {len(df):,}")
print(f"Candles in DB: {len(candles_db):,}")
print(f"Prior candle features: {len(PRIOR_CANDLE_COLS)} (21 candles × 5 OHLCV)")

Snapshots: 306,620
Candles in DB: 7,012
Prior candle features: 105 (21 candles × 5 OHLCV)


In [3]:
# Attach prior candle OHLCV to each snapshot
prior_data = []
skipped = 0
for cid in df["candle_id"].unique():
    ohlcv = get_prior_ohlcv(cid)
    if ohlcv is None:
        skipped += 1
        continue
    mask = df["candle_id"] == cid
    n = mask.sum()
    prior_data.append(
        pd.DataFrame(
            np.tile(ohlcv, (n, 1)),
            columns=PRIOR_CANDLE_COLS,
            index=df[mask].index,
        )
    )

prior_df = pd.concat(prior_data)
df = df.loc[prior_df.index].copy()  # drop candles without 21 prior
for col in PRIOR_CANDLE_COLS:
    df[col] = prior_df[col].values

# Fill NaN in all feature columns
all_indicator_cols = [c for c in df.columns if c not in {"candle_id", "session", "timestamp", "outcome", "target"}]
df[all_indicator_cols] = df[all_indicator_cols].fillna(0)

print(f"Snapshots after joining: {len(df):,} (skipped {skipped} candles without 21 prior)")
print(f"Candles: {df['candle_id'].nunique():,}")

# Define the 3 feature sets
FEATURES_A = RAW_COLS + CROSS_CANDLE_COLS  # 33: current v2
FEATURES_B = RAW_COLS + PRIOR_CANDLE_COLS  # 116: raw candle history
FEATURES_C = RAW_COLS + CROSS_CANDLE_COLS + PRIOR_CANDLE_COLS  # 138: combined

print(f"\nFeature set A (indicators): {len(FEATURES_A)}")
print(f"Feature set B (raw candle history): {len(FEATURES_B)}")
print(f"Feature set C (combined): {len(FEATURES_C)}")

Snapshots after joining: 306,620 (skipped 0 candles without 21 prior)
Candles: 6,510

Feature set A (indicators): 33
Feature set B (raw candle history): 116
Feature set C (combined): 138


/var/folders/j0/1_rf1mc97yb170c54jgv7lsr0000gn/T/ipykernel_23344/2507321467.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = prior_df[col].values


## 2. Train & Evaluate Each Feature Set

Same approach for all 3: train ContextConditionedTCN on 80% (single-snapshot), predict on ALL unseen 20% snapshots, evaluate with `run_scaling` (fees + compounding).

In [4]:
# 80/20 split
candle_ids = df["candle_id"].unique()
split = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split])
val_ids = set(candle_ids[split:])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]

# Mid-candle training data (last snapshot per candle)
df_train_mid = df_train[df_train["elapsed_pct"] <= ELAPSED_CUTOFF]
candle_train = df_train_mid.groupby("candle_id").last().reset_index()

print(f"Train: {len(train_ids)} candles | Val: {len(val_ids)} candles")
print(f"Train snapshots (mid-candle): {len(candle_train):,}")


def train_and_evaluate(name, feature_cols):
    """Train ContextConditionedTCN, evaluate with actual trading strategy."""
    raw_size = len(RAW_COLS)
    context_size = len(feature_cols) - raw_size

    # Train data
    X_tr = candle_train[feature_cols].values.astype(np.float32)
    y_tr = candle_train["target"].values.astype(np.float32)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)

    # Train model
    model = ContextConditionedTCN(raw_size=raw_size, context_size=context_size)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    X_t = torch.tensor(X_tr_s, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

    sp = int(len(X_tr_s) * 0.9)
    X_es = torch.tensor(X_tr_s[sp:], dtype=torch.float32)
    y_es = torch.tensor(y_tr[sp:], dtype=torch.float32).unsqueeze(1)

    best_vl, best_state, no_improve = float("inf"), None, 0
    t0 = time.perf_counter()
    for _epoch in range(1, 51):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss_fn(model(bx), by).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(X_es), y_es).item()
        if vl < best_vl:
            best_vl = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= 8:
            break
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    train_time = time.perf_counter() - t0

    # Predict on ALL val snapshots
    candle_data = []
    for cid in df_val["candle_id"].unique():
        snap_rows = df_val[df_val["candle_id"] == cid].sort_values("timestamp")
        if len(snap_rows) < 5:
            continue
        truth = int(snap_rows["target"].iloc[0])
        X_val = scaler.transform(snap_rows[feature_cols].fillna(0).values.astype(np.float32))
        with torch.no_grad():
            probs = torch.sigmoid(model(torch.tensor(X_val, dtype=torch.float32))).numpy().flatten()
        snapshots = []
        for i in range(len(snap_rows)):
            ua, da = snap_rows.iloc[i]["up_best_ask"], snap_rows.iloc[i]["down_best_ask"]
            snapshots.append(
                {
                    "tick": i,
                    "elapsed_pct": float(snap_rows.iloc[i]["elapsed_pct"]),
                    "pred": int(probs[i] >= 0.5),
                    "prob": float(probs[i]),
                    "up_ask": float(ua) if ua is not None and np.isfinite(ua) else None,
                    "down_ask": float(da) if da is not None and np.isfinite(da) else None,
                }
            )
        candle_data.append({"candle_id": cid, "truth": truth, "snapshots": snapshots})

    # Evaluate with strategy engine (fees + compounding)
    cfg = StrategyConfig(min_edge=0.05, max_entries=1)
    result = run_scaling(cfg, candle_data)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'=' * 60}")
    print(f"  {name} ({len(feature_cols)} features, {params:,} params)")
    print(f"{'=' * 60}")
    print(f"  Train time: {train_time:.1f}s")
    print(f"  Val candles: {len(candle_data)}")
    print(f"  Bets: {result['total_bets']}, WR: {result['win_rate'] * 100:.1f}%")
    print(f"  Return: {result['return_pct']:+.1f}%, Sharpe: {result['sharpe']:.4f}")
    print(f"  Balance: ${result['balance']:,.2f}")

    return {
        "name": name,
        "features": len(feature_cols),
        "params": params,
        "train_time": train_time,
        "bets": result["total_bets"],
        "win_rate": result["win_rate"],
        "return_pct": result["return_pct"],
        "sharpe": result["sharpe"],
        "balance": result["balance"],
    }

Train: 5208 candles | Val: 1302 candles
Train snapshots (mid-candle): 5,208


/var/folders/j0/1_rf1mc97yb170c54jgv7lsr0000gn/T/ipykernel_23344/4169708654.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  candle_train = df_train_mid.groupby("candle_id").last().reset_index()


In [5]:
# Run all 3 experiments
results = []
results.append(train_and_evaluate("A: Indicators (current v2)", FEATURES_A))
results.append(train_and_evaluate("B: Raw Candle History", FEATURES_B))
results.append(train_and_evaluate("C: Combined", FEATURES_C))


  A: Indicators (current v2) (33 features, 94,081 params)
  Train time: 81.3s
  Val candles: 1302
  Bets: 1301, WR: 50.4%
  Return: +5.0%, Sharpe: 0.0038
  Balance: $1,050.07



  B: Raw Candle History (116 features, 99,393 params)
  Train time: 81.7s
  Val candles: 1302
  Bets: 1302, WR: 51.2%
  Return: +10.0%, Sharpe: 0.0077
  Balance: $1,100.31



  C: Combined (138 features, 100,801 params)
  Train time: 80.8s
  Val candles: 1302
  Bets: 1302, WR: 49.8%
  Return: -23.5%, Sharpe: -0.0179
  Balance: $765.40


## 3. Comparison

In [6]:
import pandas as pd

comp = pd.DataFrame(results)
print(
    comp[["name", "features", "params", "bets", "win_rate", "return_pct", "sharpe", "balance"]].to_string(index=False)
)

print("\n" + "=" * 60)
best = max(results, key=lambda r: r["return_pct"])
print(f"  WINNER: {best['name']}")
print(f"  Return: {best['return_pct']:+.1f}%, WR: {best['win_rate'] * 100:.1f}%, Sharpe: {best['sharpe']:.4f}")
print("=" * 60)

                      name  features  params  bets  win_rate  return_pct    sharpe     balance
A: Indicators (current v2)        33   94081  1301  0.504228    5.007323  0.003794 1050.073225
     B: Raw Candle History       116   99393  1302  0.512289   10.030572  0.007682 1100.305725
               C: Combined       138  100801  1302  0.497696  -23.460439 -0.017931  765.395607

  WINNER: B: Raw Candle History
  Return: +10.0%, WR: 51.2%, Sharpe: 0.0077
